In [ ]:

# ==========================================
# MOBILE PRICE PREDICTION
# Using RAM, Camera, and Memory only
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Step 1: Load the original Excel dataset
df = pd.read_excel(
    r"D:\NextHikes IT Solutions  project\data set\Processed_Flipdata.xlsx"
)

# Remove spaces around column names
df.columns = df.columns.str.strip()

# Display actual column names
print("Available columns:")
print(df.columns.tolist())

# Step 2: Set feature and target column names
features = ["RAM", "Rear Camera", "Memory"]
target = "Prize"

# Check that required columns exist
required_columns = features + [target]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"These columns were not found: {missing_columns}. "
        f"Available columns: {df.columns.tolist()}"
    )

# Step 3: Convert selected columns to numeric
for col in required_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows where the target price is missing
df = df.dropna(subset=[target])

# Select only the three features
X = df[features].copy()
y = df[target].copy()

# Step 4: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# Step 5: Create the model
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Step 6: Train the model
model.fit(X_train, y_train)

print("\nModel training completed!")

# Step 7: Evaluate the model
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))


# ==========================================
# STEP 8: USER INPUT AND PRICE PREDICTION
# ==========================================

def predict_mobile_price():

    print("\n===== MOBILE PRICE PREDICTION =====")

    while True:
        try:
            ram = float(input("Enter RAM in GB: "))
            Rear_Camera = float(input("Enter Camera in MP: "))
            memory = float(input("Enter Memory/Storage in GB: "))

            values = [ram,Rear_Camera, memory]

            if not all(np.isfinite(v) and v >= 0 for v in values):
                print("Please enter valid non-negative numbers.")
                continue

            break

        except ValueError:
            print("Please enter valid numbers.")

    # Create input with the same columns used during training
    user_input = pd.DataFrame(
        [[ram,Rear_Camera, memory]],
        columns=features
    )

    # Predict price
    predicted_price = model.predict(user_input)[0]

    print("\n===================================")
    print(f"Predicted Mobile Price: ₹{predicted_price:,.2f}")
    print("===================================")


# Run the prediction function
predict_mobile_price()


In [ ]:
import pandas as pd
import numpy as np
df=pd.read_excel(r"D:\NextHikes IT Solutions  project\data set\Processed_Flipdata.xlsx")
#print(df.head)
df.shape
#print("number of missing values",df.isnull().sum())

print(df.duplicated().sum())
numerical_columns = df.select_dtypes(include=np.number).columns

for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())
    # Fill categorical columns with their most frequent value
categorical_columns = df.select_dtypes(include=['object', 'category']).columns

for col in categorical_columns:
    if df[col].isnull().any():
        if not df[col].mode().empty:
            df[col] = df[col].fillna(df[col].mode()[0])


# outliers

numeric_cols = df.select_dtypes(include="number").columns
print("numercal values",numeric_cols)
outlier_results=[]
for col in numeric_cols:
    data=df[col].dropna()

Q1=data.quantile(0.25)
Q3=data.quantile(0.75)
IQR=Q3-Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

 # Find outliers
outliers = data[(data < lower) | (data > upper)]

   # Save results
outlier_results.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Limit": lower,
        "Upper Limit": upper,
        "Outlier Count": len(outliers)
    })

outlier_df = pd.DataFrame(outlier_results)

print(outlier_df.to_string(index=False))

# Convert categorical variables

df_encoded = pd.get_dummies(
    df,
    columns=['Model', 'Colour'],
    dtype=int
)
print(df_encoded.head())

# Save the encoded dataset
df_encoded.to_csv(
    r"D:\NextHikes IT Solutions  project\data set\mobile_data_encoded.csv",
    index=False)

# Perform feature extraction 


import matplotlib.pyplot as plt
import numpy as np

numeric_df = df.select_dtypes(include=np.number)

corr = numeric_df.corr()
target = "price"
# Display correlations with price
if target in corr.columns:
    price_corr = corr[target].drop(
        target
    ).sort_values()

    print("Correlation with price:")
    print(price_corr)

    price_corr.plot(
        kind="barh",
        figsize=(9, 6),
        title="Correlation of Numerical Features with Price"
    )
    plt.xlabel("Pearson Correlation")
    plt.tight_layout()
    plt.show()

# Correlation heatmap using matplotlib
plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=90
)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

#  Model Building:


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import matplotlib.pyplot as plt

# Step 1: Load dataset
df = pd.read_csv(r"D:\NextHikes IT Solutions  project\data set\mobile_data_encoded.csv")

# Set target column
target = "Prize"

# Step 2: Clean basic data issues
df = df.drop_duplicates()

df[target] = pd.to_numeric(df[target], errors="coerce")
df = df.dropna(subset=[target])

# Separate features and target
X = df.drop(columns=[target])
y = df[target]

# Remove identifier columns if they exist
id_columns = ["id", "phone_id"]
X = X.drop(
    columns=[col for col in id_columns if col in X.columns]
)

# Identify column types
numeric_cols = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = X.select_dtypes(
    exclude=np.number
).columns.tolist()

# Step 3: Handle missing values and encode categories
numeric_pipeline = Pipeline([
("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# Step 4: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

# Step 5: Create the machine learning model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Step 6: Train the model
model.fit(X_train, y_train)

print("Model training completed!")

# Step 7: Predict prices for test data
y_pred = model.predict(X_test)

# Step 8: Evaluate model performance
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

# Step 9: Compare actual and predicted prices
results = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": y_pred
})

print("\nActual vs Predicted Prices")
print(results.head(10))

# Step 10: Visualize actual vs predicted prices
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Mobile Phone Prices")
plt.tight_layout()
plt.show()
